# Dota 2 Pro Match Predictor

## 1. Load & Explore

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
df = pd.read_csv("data/tb_pro_players_matches.csv")
df.head()

In [ ]:
print(df.shape, df.columns, df.dtypes)

In [ ]:
print(df.columns.tolist())

In [ ]:
r_cols = [c for c in df.columns if c.endswith("_r")]
d_cols = [c for c in df.columns if c.endswith("_d")]
other_cols = [c for c in df.columns if c not in r_cols and c not in d_cols]

print(f"radiant columns: {len(r_cols)}")
print(f"dire columns: {len(d_cols)}")
print(f"other columns: {len(other_cols)}")
print(other_cols)

## Drop Hero Columns

In [ ]:
hero_id_pattern = re.compile(r'^hero_\d+_avg_[rd]$')
hero_id_cols = [c for c in df.columns if hero_id_pattern.match(c)]

print(f"hero-ID columns matched: {len(hero_id_cols)}")  # 236 (118 × 2)

df.drop(columns=hero_id_cols, inplace=True)
print(df.shape)
df.head()

In [ ]:
df['radiant_win'].value_counts().plot(kind='bar', title='Radiant Win Distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df['win_pct_r'].hist(bins=30, ax=axes[0])
axes[0].set_title('win_pct_r distribution')
df['win_pct_d'].hist(bins=30, ax=axes[1])
axes[1].set_title('win_pct_d distribution')
plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(column='win_pct_r', by='radiant_win', figsize=(6,4))
plt.title('win_pct_r by match outcome')
plt.suptitle('')
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop('radiant_win', errors='ignore')
corrs = df[numeric_cols].corrwith(df['radiant_win'].astype(int)).sort_values()

corrs.tail(10).plot(kind='barh', title='Top positive correlations with radiant_win')
plt.show()
corrs.head(10).plot(kind='barh', title='Top negative correlations with radiant_win')
plt.show()

In [ ]:
plt.scatter(df['win_pct_r'], df['win_pct_d'], alpha=0.1, s=5)
plt.xlabel('win_pct_r')
plt.ylabel('win_pct_d')
plt.title('Radiant vs Dire win_pct — one point per match')
plt.show()

In [ ]:
df['win_pct_diff'] = df['win_pct_r'] - df['win_pct_d']
df['kills_per_min_diff'] = df['kills_per_min_avg_r'] - df['kills_per_min_avg_d']
df['gold_per_min_diff'] = df['gold_per_min_avg_r'] - df['gold_per_min_avg_d']

df[['win_pct_diff', 'kills_per_min_diff', 'gold_per_min_diff']].corrwith(df['radiant_win'].astype(int))

In [ ]:
df.isnull().sum()

In [ ]:
print(df[['kills_per_min_avg_r', 'kills_per_min_avg_d']].isnull().sum())
print(f"total rows: {len(df)}")

print(df[['tower_kills_avg_r', 'tower_kills_avg_d']].isnull().sum())

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
pd.set_option('display.max_rows', None)
print(missing_pct)

# bucket by severity
print("\n--- severity buckets ---")
print(f"unusable (>50% missing): {(missing_pct > 50).sum()} columns")
print(f"concerning (10-50%):     {((missing_pct > 10) & (missing_pct <= 50)).sum()} columns")
print(f"minor (0-10%):           {((missing_pct > 0) & (missing_pct <= 10)).sum()} columns")
print(f"complete:                {(missing_pct == 0).sum()} columns")

In [ ]:
parsed_cols = ['roshan_kills_avg_r', 'sentry_kills_avg_r', 'actions_per_min_avg_r', 'buyback_count_avg_r']
co_missing = df[parsed_cols].isna().sum(axis=1)
print(co_missing.value_counts())

# Dota 2 Pro Match Win Probability Predictor


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

df = pd.read_csv("data/pro_matches_raw.csv", parse_dates=["start_time"])
df = df.sort_values("start_time").reset_index(drop=True)

df["radiant_roster"] = df["radiant_roster_str"].fillna("").apply(lambda s: [int(x) for x in s.split(";") if x])
df["dire_roster"] = df["dire_roster_str"].fillna("").apply(lambda s: [int(x) for x in s.split(";") if x])

print(df.shape)
print(f"date range: {df['start_time'].min()} to {df['start_time'].max()}")
print(f"is_parsed: {df['is_parsed'].value_counts().to_dict()}")
df.head()


## 1. Feature Engineering


In [ ]:
FORM_WINDOW = 10
MIN_MATCHES = 5  # consistent with the threshold used during data collection

team_history = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
team_recent_rosters = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
h2h_history = defaultdict(lambda: deque(maxlen=FORM_WINDOW))
team_match_count = defaultdict(int)  # total appearances so far, for the MIN_MATCHES check
team_last_seen = {}  # team_id -> datetime of their most recent match

def h2h_key(a, b):
    return tuple(sorted((a, b)))

def get_form(team_id):
    hist = team_history[team_id]
    return sum(hist) / len(hist) if hist else np.nan

def get_h2h(team_a, team_b):
    key = h2h_key(team_a, team_b)
    hist = h2h_history[key]
    if not hist:
        return np.nan
    win_rate_for_first = sum(hist) / len(hist)
    return win_rate_for_first if team_a == key[0] else 1 - win_rate_for_first

def get_roster_continuity(team_id, current_roster):
    recent = team_recent_rosters[team_id]
    if not recent or not current_roster:
        return np.nan
    seen_players = set()
    for r in recent:
        seen_players.update(r)
    overlap = len(set(current_roster) & seen_players)
    return overlap / len(current_roster)

def get_rest_days(team_id, current_time):
    last_seen = team_last_seen.get(team_id)
    if last_seen is None:
        return np.nan  # first appearance for this team -- no prior match to measure rest from
    return (current_time - last_seen).total_seconds() / 86400


In [ ]:
rows = []

for row in df.itertuples():
    r_team, d_team = row.radiant_team_id, row.dire_team_id
    r_roster, d_roster = set(row.radiant_roster), set(row.dire_roster)

    rows.append({
        "match_id": row.match_id,
        "start_time": row.start_time,
        "league_name": row.league_name,
        "radiant_team_id": r_team,
        "dire_team_id": d_team,
        "radiant_form": get_form(r_team),
        "dire_form": get_form(d_team),
        "radiant_h2h_winrate": get_h2h(r_team, d_team),
        "radiant_roster_continuity": get_roster_continuity(r_team, r_roster),
        "dire_roster_continuity": get_roster_continuity(d_team, d_roster),
        "radiant_prior_matches": team_match_count[r_team],
        "dire_prior_matches": team_match_count[d_team],
        "radiant_rest_days": get_rest_days(r_team, row.start_time),
        "dire_rest_days": get_rest_days(d_team, row.start_time),
        "radiant_win": int(row.radiant_win),
        "series_type": row.series_type,
    })

    team_history[r_team].append(int(row.radiant_win))
    team_history[d_team].append(int(not row.radiant_win))
    team_recent_rosters[r_team].append(r_roster)
    team_recent_rosters[d_team].append(d_roster)
    team_match_count[r_team] += 1
    team_match_count[d_team] += 1
    team_last_seen[r_team] = row.start_time
    team_last_seen[d_team] = row.start_time
    key = h2h_key(r_team, d_team)
    h2h_history[key].append(int(row.radiant_win) if r_team == key[0] else int(not row.radiant_win))

features_df = pd.DataFrame(rows)
features_df.head(10)


## 2. Apply the match-history threshold and check missingness


In [ ]:
model_df = features_df[
    (features_df["radiant_prior_matches"] >= MIN_MATCHES) &
    (features_df["dire_prior_matches"] >= MIN_MATCHES)
].copy()

print(f"rows before threshold: {len(features_df)}, after: {len(model_df)}")
print("\nmissing values:")
print(model_df.isna().sum())

model_df["radiant_h2h_known"] = model_df["radiant_h2h_winrate"].notna().astype(int)
model_df["radiant_h2h_winrate"] = model_df["radiant_h2h_winrate"].fillna(0.5)

for side in ["radiant", "dire"]:
    col = f"{side}_rest_days"
    model_df[f"{col}_known"] = model_df[col].notna().astype(int)
    model_df[col] = model_df[col].fillna(model_df[col].median())

model_df = model_df.dropna().reset_index(drop=True)  # only drops genuinely unexpected NaNs now
print(f"\nfinal shape: {model_df.shape}")
print(f"rows with known head-to-head history: {model_df['radiant_h2h_known'].sum()} / {len(model_df)}")


### Feature: tournament tier


In [ ]:
PREMIER_LEAGUES = {"EPL Masters 2026", "PGL Wallachia 2026 Season 9"}

model_df["is_premier_league"] = model_df["league_name"].isin(PREMIER_LEAGUES).astype(int)

print(model_df.groupby("is_premier_league")["radiant_win"].agg(["mean", "count"]))


### Feature: series format (Bo1 / Bo2 / Bo3 / Bo5)


In [ ]:
SERIES_TYPE_LABELS = {0.0: "Bo1", 1.0: "Bo3", 2.0: "Bo5", 3.0: "Bo2"}
model_df["series_format"] = model_df["series_type"].map(SERIES_TYPE_LABELS)

print(model_df["series_format"].value_counts())
print(model_df.groupby("series_format")["radiant_win"].mean())

series_dummies = pd.get_dummies(model_df["series_format"], prefix="series", drop_first=True)
model_df = pd.concat([model_df, series_dummies], axis=1)
SERIES_FEATURES = series_dummies.columns.tolist()
print(f"\nnew dummy columns: {SERIES_FEATURES}")


## 3. Quick correlation check


In [ ]:
FEATURES = [
    "radiant_form", "dire_form",
    "radiant_h2h_winrate", "radiant_h2h_known",
    "radiant_roster_continuity", "dire_roster_continuity",
    "radiant_rest_days", "dire_rest_days",
    "is_premier_league",
] + SERIES_FEATURES

corrs = model_df[FEATURES].corrwith(model_df["radiant_win"]).sort_values()
corrs.plot(kind="barh", figsize=(6, 3), title="Feature correlation with radiant_win")
plt.tight_layout()
plt.show()
print(corrs)


## 4. Train/test split (time-based)


In [ ]:
model_df = model_df.sort_values("start_time").reset_index(drop=True)
split_idx = int(len(model_df) * 0.8)

train_df = model_df.iloc[:split_idx]
test_df = model_df.iloc[split_idx:]

print(f"train: {len(train_df)} ({train_df['start_time'].min()} to {train_df['start_time'].max()})")
print(f"test:  {len(test_df)} ({test_df['start_time'].min()} to {test_df['start_time'].max()})")

X_train, y_train = train_df[FEATURES], train_df["radiant_win"]
X_test, y_test = test_df[FEATURES], test_df["radiant_win"]


## 5. Baseline model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression()),
])

baseline_model.fit(X_train, y_train)
pred_proba = baseline_model.predict_proba(X_test)[:, 1]


## 6. Evaluation

In [ ]:
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

print(f"Log loss: {log_loss(y_test, pred_proba):.4f}")
print(f"AUC:      {roc_auc_score(y_test, pred_proba):.4f}")
print(f"Accuracy: {accuracy_score(y_test, pred_proba > 0.5):.4f}  (secondary metric only)")
print(f"\n(reference: always predicting 0.5 -> log loss = {log_loss(y_test, np.full_like(y_test, 0.5, dtype=float)):.4f})")


In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, pred_proba, n_bins=10)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
plt.plot(prob_pred, prob_true, marker="o", label="baseline model")
plt.xlabel("predicted win probability")
plt.ylabel("actual win rate")
plt.title("Calibration curve")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
coefs = pd.Series(baseline_model.named_steps["clf"].coef_[0], index=FEATURES).sort_values()
coefs.plot(kind="barh", figsize=(6, 3), title="Logistic regression coefficients")
plt.tight_layout()
plt.show()
